# RAG Pipeline
## Query Optimization, Reranking, and Evaluation for Stripe Documentation

### Objectives
1. **Extend previous notebooks** - Reuse vector databases and data structures
2. **Implement query transformations** - HyDE, query decomposition, multi-query
3. **Add reranking strategies** - Cross-encoder reranking, Cohere rerank
4. **Response synthesis** - Different synthesis approaches
5. **Implement RAGAS evaluation** - Automated quality metrics
6. **Compare RAG configurations** - Benchmark different pipeline setups
7. **Build production-ready patterns** - Error handling, caching, monitoring

### Prerequisites
- Completed Notebook 2 (Chunking) with chunked data available
- Completed Notebook 3 (Vector DBs) with ChromaDB populated
- OpenAI API key for embeddings and LLM
- (Optional) Cohere API key for reranking

### Environment Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# OpenAI
from openai import OpenAI


from llama_index.llms.openai import OpenAI as LlamaOpenAI


# ChromaDB (reusing from Notebook 3)
import chromadb
from chromadb.config import Settings as ChromaSettings

# Sentence transformers for reranking
from sentence_transformers import CrossEncoder
# Optional: Cohere for reranking
try:
    import cohere
    COHERE_AVAILABLE = True
except ImportError:
    COHERE_AVAILABLE = False
    print("⚠️ Cohere not available - will skip Cohere reranking")

# RAGAS for evaluation
try:
    from ragas import evaluate
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        ContextRecall
    )
    from datasets import Dataset
    RAGAS_AVAILABLE = True
except ImportError:
    RAGAS_AVAILABLE = False
    print("⚠️ RAGAS not available - will skip automated evaluation")

from dotenv import load_dotenv
load_dotenv()

print("✓ Environment setup complete")
print(f"  OpenAI API: {'✓' if os.getenv('OPENAI_API_KEY') else '✗'}")
print(f"  Cohere API: {'✓' if os.getenv('COHERE_API_KEY') else '⚠️ Optional'}")
print(f"  RAGAS: {'✓' if RAGAS_AVAILABLE else '⚠️ Optional'}")

### Configuration & Data Models

In [ ]:
# Configuration - Extends configs from previous notebooks
CONFIG = {
    # Paths (from previous notebooks)
    'data_dir': Path('./stripe_docs_data'),
    'chunks_dir': Path('./stripe_docs_data/chunks'),
    'vector_dbs_dir': Path('./stripe_docs_data/vector_dbs'),
    'output_dir': Path('./stripe_docs_data/advanced_rag'),
    
    # From Notebook 3
    'chromadb_path': './stripe_docs_data/vector_dbs/chromadb',
    'collection_name': 'stripe_docs_rag',
    'chunking_strategy': 'recursive',  # Use recursive from Notebook 2
    
    # Embedding configuration
    'embedding_model': 'text-embedding-3-small',
    'embedding_dimension': 1536,
    
    # LLM configuration
    'llm_model': 'gpt-4o-mini',
    'llm_temperature': 0.1,
    'max_tokens': 500,
    
    # Retrieval configuration
    'top_k': 5,  # Initial retrieval
    'top_k_rerank': 3,  # After reranking
    'similarity_threshold': 0.7,
    
    # Query transformation
    'use_hyde': False,  # Hypothetical Document Embeddings
    'use_multi_query': False,  # Generate multiple query variations
    'num_queries': 3,  # For multi-query approach
    
    # Reranking configuration
    'use_rerank': True,
    'rerank_model': 'cross-encoder',  # Options: 'cross-encoder', 'cohere'
    'cross_encoder_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    
    # Response synthesis
    'response_mode': 'compact',  # Options: 'refine', 'compact', 'tree_summarize', 'simple_summarize'
    
    # Evaluation
    'use_ragas': RAGAS_AVAILABLE,
    'eval_sample_size': 10,
    
    # Caching
    'cache_embeddings': True,
    'cache_dir': Path('./stripe_docs_data/cache'),
}

# Create directories
CONFIG['output_dir'].mkdir(exist_ok=True, parents=True)
CONFIG['cache_dir'].mkdir(exist_ok=True, parents=True)

print("✓ Configuration loaded")
print(f"  LLM Model: {CONFIG['llm_model']}")
print(f"  Embedding Model: {CONFIG['embedding_model']}")
print(f"  Reranking: {CONFIG['use_rerank']} ({CONFIG['rerank_model']})")
print(f"  Response Mode: {CONFIG['response_mode']}")

In [ ]:
@dataclass
class RetrievalResult:
    """Store retrieval results with metadata"""
    query: str
    documents: List[str]
    scores: List[float]
    metadata: List[Dict]
    retrieval_time: float
    reranked: bool = False
    
@dataclass
class RAGResult:
    """Complete RAG pipeline result"""
    query: str
    response: str
    source_documents: List[str]
    source_metadata: List[Dict]
    scores: List[float]
    retrieval_time: float
    generation_time: float
    total_time: float
    pipeline_config: Dict = field(default_factory=dict)
    
@dataclass
class EvaluationResult:
    """Store evaluation metrics"""
    query: str
    response: str
    ground_truth: Optional[str] = None
    faithfulness: Optional[float] = None
    answer_relevancy: Optional[float] = None
    context_relevancy: Optional[float] = None
    context_recall: Optional[float] = None
    metadata: Dict = field(default_factory=dict)

print("✓ Data models defined")

### Load Vector Database (From Notebook 3)

We'll reuse the ChromaDB instance created in Notebook 3.

In [ ]:
class ChromaDBManager:
    """Reused from Notebook 3 with enhancements for RAG"""
    
    def __init__(self, config: Dict):
        self.config = config
        self.client = chromadb.PersistentClient(
            path=config['chromadb_path'],
            settings=ChromaSettings(anonymized_telemetry=False)
        )
        self.collection_name = config['collection_name']
        
        try:
            self.collection = self.client.get_collection(name=self.collection_name)
            print(f"✓ Loaded existing collection: {self.collection_name}")
            print(f"  Documents: {self.collection.count()}")
        except:
            raise ValueError(
                f"Collection '{self.collection_name}' not found. "
                "Please run Notebook 3 first to create the vector database."
            )
    
    def query(
        self,
        query_embedding: List[float],
        n_results: int = 5,
        where: Optional[Dict] = None
    ) -> 'RetrievalResult':
        """Query the vector database"""
        start_time = time.time()
        
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where
        )
        
        retrieval_time = time.time() - start_time
        
        # Extract results
        documents = results['documents'][0] if results['documents'] else []
        distances = results['distances'][0] if results['distances'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        
        # Convert distances to similarity scores (1 - distance for cosine)
        scores = [1 - d for d in distances]
        
        return RetrievalResult(
            query="",  # Will be set by caller
            documents=documents,
            scores=scores,
            metadata=metadatas,
            retrieval_time=retrieval_time,
            reranked=False
        )
    
    def get_stats(self) -> Dict:
        """Get collection statistics"""
        return {
            'collection_name': self.collection_name,
            'total_documents': self.collection.count(),
            'path': self.config['chromadb_path']
        }

# Initialize ChromaDB manager
chroma_manager = ChromaDBManager(CONFIG)
stats = chroma_manager.get_stats()

print("\nDatabase Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

### Embedding and LLM Setup

In [ ]:
class EmbeddingManager:
    """Manage embeddings with caching"""
    
    def __init__(self, config: Dict):
        self.config = config
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = config['embedding_model']
        self.cache_dir = config['cache_dir'] / 'embeddings'
        self.cache_dir.mkdir(exist_ok=True, parents=True)
        
        # Statistics
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _get_cache_key(self, text: str) -> str:
        """Generate cache key from text"""
        return hashlib.md5(f"{self.model}:{text}".encode()).hexdigest()
    
    def get_embedding(self, text: str, use_cache: bool = True) -> List[float]:
        """Get embedding with optional caching"""
        if use_cache and self.config['cache_embeddings']:
            cache_key = self._get_cache_key(text)
            cache_file = self.cache_dir / f"{cache_key}.json"
            
            if cache_file.exists():
                with open(cache_file, 'r') as f:
                    self.cache_hits += 1
                    return json.load(f)
        
        # Generate embedding
        response = self.client.embeddings.create(
            model=self.model,
            input=text
        )
        embedding = response.data[0].embedding
        
        # Cache if enabled
        if use_cache and self.config['cache_embeddings']:
            cache_key = self._get_cache_key(text)
            cache_file = self.cache_dir / f"{cache_key}.json"
            with open(cache_file, 'w') as f:
                json.dump(embedding, f)
            self.cache_misses += 1
        
        return embedding
    
    def get_cache_stats(self) -> Dict:
        """Get caching statistics"""
        total = self.cache_hits + self.cache_misses
        hit_rate = (self.cache_hits / total * 100) if total > 0 else 0
        return {
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'hit_rate': f"{hit_rate:.1f}%"
        }

# Initialize embedding manager
embedding_manager = EmbeddingManager(CONFIG)

# Initialize LLM
llm = LlamaOpenAI(
    model=CONFIG['llm_model'],
    temperature=CONFIG['llm_temperature'],
    max_tokens=CONFIG['max_tokens']
)

print("✓ Embedding manager initialized")
print(f"  Model: {CONFIG['embedding_model']}")
print(f"  Caching: {'Enabled' if CONFIG['cache_embeddings'] else 'Disabled'}")
print("✓ LLM initialized")
print(f"  Model: {CONFIG['llm_model']}")

### Query Transformation Strategies

In [ ]:
class QueryTransformer:
    """Transform queries for better retrieval"""
    
    def __init__(self, llm, embedding_manager: EmbeddingManager, config: Dict):
        self.llm = llm
        self.embedding_manager = embedding_manager
        self.config = config
    
    def generate_hyde_document(self, query: str) -> str:
        """
        HyDE: Generate a hypothetical document that would answer the query.
        Use this document's embedding for retrieval instead of the query.
        """
        prompt = f"""Generate a detailed, technical answer to this question about Stripe API.
Write as if you're documentation that would answer this question.

Question: {query}

Answer:"""
        
        response = self.llm.complete(prompt)
        return response.text
    
    def generate_multi_queries(self, query: str, num_queries: int = 3) -> List[str]:
        """
        Generate multiple variations of the query for better coverage.
        """
        prompt = f"""Generate {num_queries} different versions of this question about Stripe API.
Each version should ask the same thing but with different wording.

Original question: {query}

Generate {num_queries} variations (one per line):"""
        
        response = self.llm.complete(prompt)
        variations = [q.strip() for q in response.text.strip().split('\n') if q.strip()]
        
        # Include original query
        return [query] + variations[:num_queries-1]
    
    def decompose_query(self, query: str) -> List[str]:
        """
        Break down complex queries into simpler sub-queries.
        """
        prompt = f"""Break down this complex question into 2-3 simpler sub-questions.

Question: {query}

Sub-questions (one per line):"""
        
        response = self.llm.complete(prompt)
        sub_queries = [q.strip() for q in response.text.strip().split('\n') if q.strip()]
        return sub_queries

# Initialize query transformer
query_transformer = QueryTransformer(llm, embedding_manager, CONFIG)

# Test query transformation
test_query = "How do I create a payment intent with automatic payment methods?"

print("Testing Query Transformations:")
print(f"\nOriginal Query: {test_query}")

if CONFIG['use_hyde']:
    print("\n--- HyDE Document ---")
    hyde_doc = query_transformer.generate_hyde_document(test_query)
    print(hyde_doc[:200] + "...")

if CONFIG['use_multi_query']:
    print("\n--- Multi-Query Variations ---")
    variations = query_transformer.generate_multi_queries(test_query, CONFIG['num_queries'])
    for i, var in enumerate(variations, 1):
        print(f"{i}. {var}")

print("\n✓ Query transformation ready")

### Reranking Strategies

In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

class Reranker:
    """Rerank retrieved documents for better relevance"""
    
    def __init__(self, config: Dict):
        self.config = config
        self.rerank_type = config['rerank_model']
        
        if self.rerank_type == 'cross-encoder':
            print(f"Loading cross-encoder model: {config['cross_encoder_model']}")
            self.cross_encoder = CrossEncoder(config['cross_encoder_model'])
            print("✓ Cross-encoder loaded")
        
        elif self.rerank_type == 'cohere' and COHERE_AVAILABLE:
            self.cohere_client = cohere.Client(os.getenv('COHERE_API_KEY'))
            print("✓ Cohere client initialized")
        
    def rerank_cross_encoder(
        self,
        query: str,
        documents: List[str],
        metadata: List[Dict],
        top_k: int
    ) -> Tuple[List[str], List[float], List[Dict]]:
        """Rerank using cross-encoder model"""
        # Create pairs for cross-encoder
        pairs = [[query, doc] for doc in documents]
        
        # Get scores
        scores = self.cross_encoder.predict(pairs)
        
        # Sort by score
        sorted_indices = np.argsort(scores)[::-1][:top_k]
        
        reranked_docs = [documents[i] for i in sorted_indices]
        reranked_scores = [float(scores[i]) for i in sorted_indices]
        reranked_metadata = [metadata[i] for i in sorted_indices]
        
        return reranked_docs, reranked_scores, reranked_metadata
    
    def rerank_cohere(
        self,
        query: str,
        documents: List[str],
        metadata: List[Dict],
        top_k: int
    ) -> Tuple[List[str], List[float], List[Dict]]:
        """Rerank using Cohere's rerank API"""
        response = self.cohere_client.rerank(
            model='rerank-english-v2.0',
            query=query,
            documents=documents,
            top_n=top_k
        )
        
        reranked_docs = [documents[r.index] for r in response.results]
        reranked_scores = [r.relevance_score for r in response.results]
        reranked_metadata = [metadata[r.index] for r in response.results]
        
        return reranked_docs, reranked_scores, reranked_metadata
    
    def rerank(
        self,
        query: str,
        documents: List[str],
        metadata: List[Dict],
        top_k: Optional[int] = None
    ) -> Tuple[List[str], List[float], List[Dict]]:
        """Rerank documents using configured method"""
        if top_k is None:
            top_k = self.config['top_k_rerank']
        
        if self.rerank_type == 'cross-encoder':
            return self.rerank_cross_encoder(query, documents, metadata, top_k)
        elif self.rerank_type == 'cohere':
            return self.rerank_cohere(query, documents, metadata, top_k)
        else:
            # No reranking, just return top_k
            return documents[:top_k], [1.0] * min(top_k, len(documents)), metadata[:top_k]

# Initialize reranker if enabled
reranker = Reranker(CONFIG) if CONFIG['use_rerank'] else None

if reranker:
    print(f"✓ Reranker initialized ({CONFIG['rerank_model']})")
else:
    print("⚠️ Reranking disabled")

### RAG Pipeline

In [ ]:
class AdvancedRAGPipeline:
    """Complete advanced RAG pipeline with all components"""
    
    def __init__(
        self,
        chroma_manager: ChromaDBManager,
        embedding_manager: EmbeddingManager,
        query_transformer: QueryTransformer,
        reranker: Optional[Reranker],
        llm,
        config: Dict
    ):
        self.chroma_manager = chroma_manager
        self.embedding_manager = embedding_manager
        self.query_transformer = query_transformer
        self.reranker = reranker
        self.llm = llm
        self.config = config
    
    def retrieve(
        self,
        query: str,
        top_k: Optional[int] = None,
        use_query_transform: bool = False,
        metadata_filter: Optional[Dict] = None
    ) -> RetrievalResult:
        """Retrieve relevant documents"""
        if top_k is None:
            top_k = self.config['top_k']
        
        # Query transformation
        query_text = query
        if use_query_transform and self.config['use_hyde']:
            hyde_doc = self.query_transformer.generate_hyde_document(query)
            query_text = hyde_doc
        
        # Get embedding
        query_embedding = self.embedding_manager.get_embedding(query_text)
        
        # Retrieve from vector DB
        result = self.chroma_manager.query(
            query_embedding=query_embedding,
            n_results=top_k,
            where=metadata_filter
        )
        result.query = query  # Set original query
        
        return result
    
    def retrieve_and_rerank(
        self,
        query: str,
        initial_k: Optional[int] = None,
        final_k: Optional[int] = None,
        use_query_transform: bool = False,
        metadata_filter: Optional[Dict] = None
    ) -> RetrievalResult:
        """Retrieve and rerank for better results"""
        if initial_k is None:
            initial_k = self.config['top_k']
        if final_k is None:
            final_k = self.config['top_k_rerank']
        
        # Initial retrieval
        retrieval_result = self.retrieve(
            query=query,
            top_k=initial_k,
            use_query_transform=use_query_transform,
            metadata_filter=metadata_filter
        )
        
        # Rerank if enabled
        if self.reranker and self.config['use_rerank']:
            start_time = time.time()
            reranked_docs, reranked_scores, reranked_metadata = self.reranker.rerank(
                query=query,
                documents=retrieval_result.documents,
                metadata=retrieval_result.metadata,
                top_k=final_k
            )
            rerank_time = time.time() - start_time
            
            return RetrievalResult(
                query=query,
                documents=reranked_docs,
                scores=reranked_scores,
                metadata=reranked_metadata,
                retrieval_time=retrieval_result.retrieval_time + rerank_time,
                reranked=True
            )
        
        return retrieval_result
    
    def generate_response(
        self,
        query: str,
        context_docs: List[str],
        response_mode: Optional[str] = None
    ) -> Tuple[str, float]:
        """Generate response using retrieved context"""
        if response_mode is None:
            response_mode = self.config['response_mode']
        
        # Build context
        context = "\n\n".join([
            f"Document {i+1}:\n{doc}"
            for i, doc in enumerate(context_docs)
        ])
        
        # Create prompt
        prompt = f"""You are a helpful assistant answering questions about Stripe API.

Context from documentation:
{context}

Question: {query}

Please provide a clear, accurate answer based on the context above. If the context doesn't contain enough information, say so.

Answer:"""
        
        start_time = time.time()
        response = self.llm.complete(prompt)
        generation_time = time.time() - start_time
        
        return response.text, generation_time
    
    def query(
        self,
        query: str,
        use_rerank: Optional[bool] = None,
        use_query_transform: bool = False,
        metadata_filter: Optional[Dict] = None
    ) -> RAGResult:
        """Complete RAG pipeline: retrieve, rerank, generate"""
        if use_rerank is None:
            use_rerank = self.config['use_rerank']
        
        start_time = time.time()
        
        # Retrieve (and optionally rerank)
        if use_rerank:
            retrieval_result = self.retrieve_and_rerank(
                query=query,
                use_query_transform=use_query_transform,
                metadata_filter=metadata_filter
            )
        else:
            retrieval_result = self.retrieve(
                query=query,
                use_query_transform=use_query_transform,
                metadata_filter=metadata_filter
            )
        
        # Generate response
        response_text, generation_time = self.generate_response(
            query=query,
            context_docs=retrieval_result.documents
        )
        
        total_time = time.time() - start_time
        
        return RAGResult(
            query=query,
            response=response_text,
            source_documents=retrieval_result.documents,
            source_metadata=retrieval_result.metadata,
            scores=retrieval_result.scores,
            retrieval_time=retrieval_result.retrieval_time,
            generation_time=generation_time,
            total_time=total_time,
            pipeline_config={
                'reranked': retrieval_result.reranked,
                'query_transform': use_query_transform,
                'metadata_filter': metadata_filter is not None
            }
        )

# Initialize RAG pipeline
rag_pipeline = AdvancedRAGPipeline(
    chroma_manager=chroma_manager,
    embedding_manager=embedding_manager,
    query_transformer=query_transformer,
    reranker=reranker,
    llm=llm,
    config=CONFIG
)

print("✓ Advanced RAG Pipeline initialized")
print("  Components:")
print(f"    - Vector DB: ChromaDB")
print(f"    - Embeddings: {CONFIG['embedding_model']}")
print(f"    - LLM: {CONFIG['llm_model']}")
print(f"    - Reranking: {CONFIG['use_rerank']}")
print(f"    - Query Transform: {CONFIG['use_hyde'] or CONFIG['use_multi_query']}")

### Test Queries and Evaluation

In [ ]:
# Define test queries
TEST_QUERIES = [
    "How do I create a payment intent?",
    "What's the difference between Payment Intents and Charges?",
    "How do I handle webhook events for successful payments?",
    "What payment methods are supported by Stripe?",
    "How do I set up recurring billing with subscriptions?",
    "What's the process for handling refunds?",
    "How do I test my integration in development?",
    "What are the best practices for error handling?",
    "How do I implement 3D Secure authentication?",
    "What's the recommended way to store customer payment methods?"
]

print(f"Test queries defined: {len(TEST_QUERIES)}")

In [ ]:
# Run test queries
print("Running test queries through RAG pipeline...\n")

results = []

for i, query in enumerate(tqdm(TEST_QUERIES[:3], desc="Processing queries"), 1):  # Start with 3 for testing
    print(f"\n{'='*80}")
    print(f"Query {i}: {query}")
    print(f"{'='*80}")
    
    # Run RAG pipeline
    result = rag_pipeline.query(query)
    results.append(result)
    
    # Display results
    print(f"\nResponse:")
    print(result.response)
    
    print(f"\nPerformance:")
    print(f"  Retrieval time: {result.retrieval_time*1000:.1f}ms")
    print(f"  Generation time: {result.generation_time*1000:.1f}ms")
    print(f"  Total time: {result.total_time*1000:.1f}ms")
    
    print(f"\nSources (top {len(result.source_documents)}):")
    for j, (doc, score, metadata) in enumerate(zip(
        result.source_documents,
        result.scores,
        result.source_metadata
    ), 1):
        doc_type = metadata.get('doc_type', 'N/A')
        category = metadata.get('category', 'N/A')
        print(f"  {j}. [{doc_type}/{category}] Score: {score:.3f}")
        print(f"     {doc[:100]}...")

print(f"\n✓ Completed {len(results)} test queries")

### Compare RAG Configurations

In [ ]:
# Compare different RAG configurations
print("Comparing RAG configurations...\n")

test_query = "How do I create a payment intent?"

configurations = [
    {'name': 'Baseline', 'use_rerank': False, 'use_query_transform': False},
    {'name': 'With Reranking', 'use_rerank': True, 'use_query_transform': False},
    {'name': 'With HyDE', 'use_rerank': False, 'use_query_transform': True},
    {'name': 'Full Pipeline', 'use_rerank': True, 'use_query_transform': True},
]

comparison_results = []

for config in configurations:
    print(f"\nTesting: {config['name']}")
    
    # Temporarily update config
    original_rerank = CONFIG['use_rerank']
    original_hyde = CONFIG['use_hyde']
    
    result = rag_pipeline.query(
        query=test_query,
        use_rerank=config['use_rerank'],
        use_query_transform=config['use_query_transform']
    )
    
    comparison_results.append({
        'configuration': config['name'],
        'retrieval_time_ms': result.retrieval_time * 1000,
        'generation_time_ms': result.generation_time * 1000,
        'total_time_ms': result.total_time * 1000,
        'avg_score': np.mean(result.scores),
        'response_length': len(result.response)
    })
    
    print(f"  Total time: {result.total_time*1000:.1f}ms")
    print(f"  Avg score: {np.mean(result.scores):.3f}")

# Display comparison
comparison_df = pd.DataFrame(comparison_results)
print("\nConfiguration Comparison:")
print(comparison_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time comparison
comparison_df.plot(x='configuration', y='total_time_ms', kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Time by Configuration')
axes[0].set_ylabel('Time (ms)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Score comparison
comparison_df.plot(x='configuration', y='avg_score', kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Average Relevance Score by Configuration')
axes[1].set_ylabel('Score')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'rag_configuration_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Configuration comparison complete")

### RAGAS Evaluation

Automated evaluation using RAGAS framework.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import MetricWithEmbeddings
from ragas.metrics import MetricWithLLM
from ragas import RunConfig



if RAGAS_AVAILABLE and CONFIG['use_ragas']:
    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextRecall()
    ]

    llm = ChatOpenAI(model=CONFIG['llm_model'])
    emb = OpenAIEmbeddings()
    embedding=LangchainEmbeddingsWrapper(emb),


    for metric in metrics:
        if isinstance(metric, MetricWithLLM):
            metric.llm = llm
        if isinstance(metric, MetricWithEmbeddings):
            metric.embeddings = embedding
        run_config = RunConfig()
        metric.init(run_config)

    print("Running RAGAS evaluation...\n")
    
    # Prepare evaluation data
    eval_data = {
        'question': [],
        'answer': [],
        'contexts': [],
        'ground_truth': []  # Optional, if you have ground truth
    }
    
    # Use a subset of test queries
    eval_queries = TEST_QUERIES[:CONFIG['eval_sample_size']]
    
    for query in tqdm(eval_queries[:3], desc="Generating responses for evaluation"):
        result = rag_pipeline.query(query)
        
        eval_data['question'].append(query)
        eval_data['answer'].append(result.response)
        eval_data['contexts'].append(result.source_documents)
        eval_data['ground_truth'].append("")  # Add if available
    
    # Create dataset
    dataset = Dataset.from_dict(eval_data)
    
    # Run evaluation
    print("\nEvaluating with RAGAS metrics...")

    
    result = evaluate(dataset=dataset, metrics=metrics, llm = llm)
    print(result)

    # Save results
    with open(CONFIG['output_dir'] / 'ragas_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print("\n✓ RAGAS evaluation complete")
else:
    print("⚠️ RAGAS evaluation skipped (not available or disabled)")

### Performance Analysis

In [ ]:
# Analyze performance across all results
print("Performance Analysis\n" + "="*80)

if results:
    # Extract metrics
    retrieval_times = [r.retrieval_time * 1000 for r in results]
    generation_times = [r.generation_time * 1000 for r in results]
    total_times = [r.total_time * 1000 for r in results]
    avg_scores = [np.mean(r.scores) for r in results]
    
    stats_df = pd.DataFrame({
        'Metric': ['Retrieval Time (ms)', 'Generation Time (ms)', 'Total Time (ms)', 'Avg Relevance Score'],
        'Mean': [
            np.mean(retrieval_times),
            np.mean(generation_times),
            np.mean(total_times),
            np.mean(avg_scores)
        ],
        'Std': [
            np.std(retrieval_times),
            np.std(generation_times),
            np.std(total_times),
            np.std(avg_scores)
        ],
        'Min': [
            np.min(retrieval_times),
            np.min(generation_times),
            np.min(total_times),
            np.min(avg_scores)
        ],
        'Max': [
            np.max(retrieval_times),
            np.max(generation_times),
            np.max(total_times),
            np.max(avg_scores)
        ]
    })
    
    print(stats_df.to_string(index=False))
    
    # Visualize performance distribution
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].hist(retrieval_times, bins=15, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Retrieval Time Distribution')
    axes[0, 0].set_xlabel('Time (ms)')
    axes[0, 0].set_ylabel('Frequency')
    
    axes[0, 1].hist(generation_times, bins=15, color='lightcoral', edgecolor='black')
    axes[0, 1].set_title('Generation Time Distribution')
    axes[0, 1].set_xlabel('Time (ms)')
    axes[0, 1].set_ylabel('Frequency')
    
    axes[1, 0].hist(total_times, bins=15, color='lightgreen', edgecolor='black')
    axes[1, 0].set_title('Total Time Distribution')
    axes[1, 0].set_xlabel('Time (ms)')
    axes[1, 0].set_ylabel('Frequency')
    
    axes[1, 1].hist(avg_scores, bins=15, color='plum', edgecolor='black')
    axes[1, 1].set_title('Relevance Score Distribution')
    axes[1, 1].set_xlabel('Score')
    axes[1, 1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'performance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Cache statistics
    print("\nEmbedding Cache Statistics:")
    cache_stats = embedding_manager.get_cache_stats()
    for key, value in cache_stats.items():
        print(f"  {key}: {value}")
    
    print("\n✓ Performance analysis complete")
else:
    print("⚠️ No results to analyze yet")

### Save Results and Export

In [ ]:
# Save all results
print("Saving results...\n")

# Convert results to JSON-serializable format
results_data = []
for r in results:
    results_data.append({
        'query': r.query,
        'response': r.response,
        'source_documents': r.source_documents,
        'source_metadata': r.source_metadata,
        'scores': r.scores,
        'retrieval_time': r.retrieval_time,
        'generation_time': r.generation_time,
        'total_time': r.total_time,
        'pipeline_config': r.pipeline_config
    })

# Save results
with open(CONFIG['output_dir'] / 'rag_results.json', 'w') as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to {CONFIG['output_dir'] / 'rag_results.json'}")

# Save configuration
config_export = CONFIG.copy()
config_export['data_dir'] = str(config_export['data_dir'])
config_export['chunks_dir'] = str(config_export['chunks_dir'])
config_export['vector_dbs_dir'] = str(config_export['vector_dbs_dir'])
config_export['output_dir'] = str(config_export['output_dir'])
config_export['cache_dir'] = str(config_export['cache_dir'])

with open(CONFIG['output_dir'] / 'pipeline_config.json', 'w') as f:
    json.dump(config_export, f, indent=2)

print(f"✓ Configuration saved to {CONFIG['output_dir'] / 'pipeline_config.json'}")

# Save comparison results if available
if 'comparison_df' in locals():
    comparison_df.to_csv(CONFIG['output_dir'] / 'configuration_comparison.csv', index=False)
    print(f"✓ Comparison saved to {CONFIG['output_dir'] / 'configuration_comparison.csv'}")

print("\n✓ All results saved")